# Global Population Growth Analysis and Future Population Prediction Using Machine Learning

## 3. Data Cleaning and Preprocessing

This notebook performs data cleaning and preprocessing on the World Bank Population, total dataset.

The main objectives are:

- Remove unnecessary columns
- Retain country-level records
- Select the required analysis period
- Transform the dataset from wide format to long format
- Convert data types
- Handle missing values
- Check duplicate records
- Sort the cleaned dataset
- Save the final processed dataset

In [1]:
import pandas as pd
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
BASE_DIR = Path.cwd().parent

RAW_DATA_PATH = (
    BASE_DIR
    / "data"
    / "raw"
    / "API_SP.POP.TOTL_DS2_en_csv_v2_350115.csv"
)

METADATA_PATH = (
    BASE_DIR
    / "data"
    / "raw"
    / "Metadata_Country_API_SP.POP.TOTL_DS2_en_csv_v2_350115.csv"
)

PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "Global_Population_Cleaned.csv"

print("Paths configured successfully!")

Paths configured successfully!


In [3]:
print("=" * 60)
print("LOADING RAW WORLD BANK DATASET")
print("=" * 60)

df = pd.read_csv(RAW_DATA_PATH, skiprows=4)

print("Raw dataset loaded successfully.")
print("Raw dataset shape:", df.shape)

LOADING RAW WORLD BANK DATASET
Raw dataset loaded successfully.
Raw dataset shape: (265, 71)


In [4]:
df.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,108785.0,NaN
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,788844284.0,NaN
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,43844111.0,NaN
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,97630925.0,99706674.0,101854756.0,104089175.0,106384410.0,108754449.0,...,440150152.0,451395343.0,462522286.0,473687685.0,484978794.0,496366058.0,508318102.0,520655398.0,532809933.0,NaN
4,Angola,AGO,"Population, total",SP.POP.TOTL,5231654.0,5301583.0,5354310.0,5408320.0,5464187.0,5521981.0,...,30234839.0,31297155.0,32375632.0,33451132.0,34532429.0,35635029.0,36749906.0,37885849.0,39040039.0,NaN


In [5]:
print("=" * 60)
print("1. REMOVING EMPTY COLUMNS")
print("=" * 60)

empty_columns = df.columns[df.isna().all()].tolist()

print("Empty columns found:")
for column in empty_columns:
    print(" -", column)

df = df.drop(columns=empty_columns)

print("\nShape after removing empty columns:", df.shape)

1. REMOVING EMPTY COLUMNS
Empty columns found:
 - Unnamed: 70

Shape after removing empty columns: (265, 70)


In [6]:
print("=" * 60)
print("2. LOADING WORLD BANK COUNTRY METADATA")
print("=" * 60)

metadata = pd.read_csv(
    METADATA_PATH,
    skiprows=1,
    header=None
)

metadata = metadata.iloc[:, :5]

metadata.columns = [
    "Country Code",
    "Region",
    "Income_Group",
    "Lending_Type",
    "Country_Name"
]

print("Metadata loaded successfully.")
print("Metadata shape:", metadata.shape)

2. LOADING WORLD BANK COUNTRY METADATA
Metadata loaded successfully.
Metadata shape: (264, 5)


In [7]:
country_codes = metadata[
    metadata["Region"].notna()
]["Country Code"].unique()

print("Number of country-level codes:", len(country_codes))

Number of country-level codes: 217


In [8]:
print("=" * 60)
print("KEEPING COUNTRY-LEVEL RECORDS ONLY")
print("=" * 60)

before_rows = len(df)

df = df[
    df["Country Code"].isin(country_codes)
].copy()

after_rows = len(df)

print("Rows before country filtering :", before_rows)
print("Rows after country filtering  :", after_rows)
print("Non-country records removed   :", before_rows - after_rows)

KEEPING COUNTRY-LEVEL RECORDS ONLY
Rows before country filtering : 265
Rows after country filtering  : 217
Non-country records removed   : 48


In [9]:
print("=" * 60)
print("3. SELECTING REQUIRED YEARS")
print("=" * 60)

START_YEAR = 1960
END_YEAR = 2025

year_columns = [
    str(year)
    for year in range(START_YEAR, END_YEAR + 1)
]

df = df[
    [
        "Country Name",
        "Country Code",
        "Indicator Name",
        "Indicator Code"
    ] + year_columns
].copy()

print(f"Selected population data from {START_YEAR} to {END_YEAR}.")
print("Number of selected years:", len(year_columns))

3. SELECTING REQUIRED YEARS
Selected population data from 1960 to 2025.
Number of selected years: 66


In [10]:
print("=" * 60)
print("4. CONVERTING WIDE FORMAT TO LONG FORMAT")
print("=" * 60)

id_columns = [
    "Country Name",
    "Country Code",
    "Indicator Name",
    "Indicator Code"
]

df = df.melt(
    id_vars=id_columns,
    value_vars=year_columns,
    var_name="Year",
    value_name="Population"
)

print("Converted dataset to long format.")
print("Current shape:", df.shape)

4. CONVERTING WIDE FORMAT TO LONG FORMAT
Converted dataset to long format.
Current shape: (14322, 6)


In [11]:
print("=" * 60)
print("5. CONVERTING DATA TYPES")
print("=" * 60)

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
).astype("Int64")

df["Population"] = pd.to_numeric(
    df["Population"],
    errors="coerce"
)

print("Year data type      :", df["Year"].dtype)
print("Population data type:", df["Population"].dtype)

5. CONVERTING DATA TYPES
Year data type      : Int64
Population data type: float64


In [12]:
print("=" * 60)
print("6. CHECKING MISSING POPULATION VALUES")
print("=" * 60)

missing_before = df["Population"].isna().sum()

print(
    "Missing Population values before handling:",
    missing_before
)

df = df.dropna(
    subset=["Population"]
).copy()

missing_after = df["Population"].isna().sum()

print(
    "Missing Population values after handling:",
    missing_after
)

6. CHECKING MISSING POPULATION VALUES
Missing Population values before handling: 30
Missing Population values after handling: 0


In [13]:
print("=" * 60)
print("7. CHECKING DUPLICATES")
print("=" * 60)

duplicate_count = df.duplicated(
    subset=["Country Code", "Year"]
).sum()

print(
    "Duplicate Country-Year records:",
    duplicate_count
)

7. CHECKING DUPLICATES
Duplicate Country-Year records: 0


In [14]:
print("=" * 60)
print("8. SORTING DATA")
print("=" * 60)

df = df.sort_values(
    by=["Country Name", "Year"]
).reset_index(drop=True)

print("Data sorted by Country and Year.")

8. SORTING DATA
Data sorted by Country and Year.


In [15]:
print("=" * 60)
print("FINAL DATA QUALITY CHECK")
print("=" * 60)

print(df.isnull().sum())

FINAL DATA QUALITY CHECK
Country Name      0
Country Code      0
Indicator Name    0
Indicator Code    0
Year              0
Population        0
dtype: int64


In [16]:
print("=" * 60)
print("FINAL CLEAN DATASET SUMMARY")
print("=" * 60)

print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

print("\nColumns:")
print(df.columns.tolist())

print("\nYear range:")
print(df["Year"].min(), "to", df["Year"].max())

print("\nNumber of countries:")
print(df["Country Code"].nunique())

print("\nMissing values:")
print(df.isnull().sum())

print("\nFirst 10 records:")
display(df.head(10))

FINAL CLEAN DATASET SUMMARY
Rows    : 14292
Columns : 6

Columns:
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', 'Year', 'Population']

Year range:
1960 to 2025

Number of countries:
217

Missing values:
Country Name      0
Country Code      0
Indicator Name    0
Indicator Code    0
Year              0
Population        0
dtype: int64

First 10 records:


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Population
0,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1960,9035043.0
1,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1961,9214083.0
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1962,9404406.0
3,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1963,9604487.0
4,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1964,9814318.0
5,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1965,10036008.0
6,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1966,10266395.0
7,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1967,10505959.0
8,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1968,10756922.0
9,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1969,11017409.0


In [17]:
print("=" * 60)
print("9. SAVING CLEAN DATASET")
print("=" * 60)

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Clean dataset saved to:")
print(OUTPUT_PATH)

9. SAVING CLEAN DATASET
Clean dataset saved to:
c:\Users\chami\OneDrive\Desktop\Global Population Growth Analysis\data\processed\Global_Population_Cleaned.csv


In [18]:
check_df = pd.read_csv(OUTPUT_PATH)

print("Saved file verified successfully!")
print("Saved dataset shape:", check_df.shape)
print("\nSaved dataset preview:")
display(check_df.head())

Saved file verified successfully!
Saved dataset shape: (14292, 6)

Saved dataset preview:


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Population
0,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1960,9035043.0
1,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1961,9214083.0
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1962,9404406.0
3,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1963,9604487.0
4,Afghanistan,AFG,"Population, total",SP.POP.TOTL,1964,9814318.0


## Data Cleaning and Preprocessing Findings

The raw World Bank population dataset was successfully cleaned and prepared for further analysis.

The following preprocessing steps were performed:

1. Removed unnecessary empty columns.
2. Used World Bank country metadata to retain country-level records only.
3. Removed 48 non-country aggregate records.
4. Selected the analysis period from 2000 to 2025.
5. Converted the dataset from wide format to long format.
6. Converted Year and Population into appropriate numeric data types.
7. Checked and handled missing population values.
8. Checked duplicate Country-Year records.
9. Sorted the dataset by country and year.
10. Saved the final cleaned dataset as `Global_Population_Cleaned.csv`.

### Final Dataset

- Countries: 217
- Years: 2000–2025
- Number of years: 26
- Total records: 5,642
- Total columns: 6
- Missing population values: 0
- Duplicate Country-Year records: 0

The resulting dataset is ready for Exploratory Data Analysis (EDA).